# 02 — Data Cleaning: Titik Panas Kebakaran Indonesia (NASA FIRMS VIIRS S-NPP)

Notebook ini membersihkan data mentah hasil profiling (notebook 01)
menggunakan **Polars lazy API**, memvalidasi hasilnya, menyimpannya
dalam format **Parquet**, lalu memprofilkan ulang dengan
**DuckDB `SUMMARIZE`**.

Setiap tahap pipeline disimpan dalam variabel terpisah agar notebook
dapat dijalankan ulang tanpa bergantung pada urutan eksekusi.

In [1]:
from pathlib import Path
import polars as pl
import duckdb

ROOT = Path("/home/jovyan/work")
DATA_PATH = str(ROOT / "data/raw/firms/*.csv")
OUT_PATH = ROOT / "data/processed/firms_clean.parquet"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Membaca Data dengan Schema Eksplisit

Profiling menunjukkan Polars dan DuckDB menebak tipe `acq_date` dan
`acq_time` secara berbeda. Karena itu tipe ditetapkan secara eksplisit:
`acq_date` sebagai `Date`, dan `acq_time` sebagai `String` agar nol di
depan (misalnya `0308`) tidak hilang.

In [2]:
lf_raw = pl.scan_csv(
    DATA_PATH,
    schema_overrides={"acq_date": pl.Date, "acq_time": pl.String},
)
n_raw = lf_raw.select(pl.len()).collect().item()
print(f"Jumlah baris mentah: {n_raw:,}")

Jumlah baris mentah: 3,418,464


## 2. Transformasi Waktu

- `dt_utc`: gabungan `acq_date` dan `acq_time` (UTC).
- `dt_local_solar`: waktu matahari lokal, dihitung sebagai
  UTC + bujur/15 jam. Diperlukan karena deteksi malam (±17:00–20:00 UTC)
  jatuh pada dini hari tanggal berikutnya di Indonesia.
- `date_local`, `year`, `month`: diturunkan dari waktu lokal.

In [3]:
lf_time = (
    lf_raw
    .with_columns(
        (
            pl.col("acq_date").cast(pl.Datetime)
            + pl.duration(
                hours=pl.col("acq_time").str.slice(0, 2).cast(pl.Int32),
                minutes=pl.col("acq_time").str.slice(2, 2).cast(pl.Int32),
            )
        ).alias("dt_utc")
    )
    .with_columns(
        (
            pl.col("dt_utc")
            + pl.duration(minutes=(pl.col("longitude") / 15 * 60).round().cast(pl.Int64))
        ).alias("dt_local_solar")
    )
    .with_columns(
        pl.col("dt_local_solar").dt.date().alias("date_local"),
        pl.col("dt_local_solar").dt.year().alias("year"),
        pl.col("dt_local_solar").dt.month().alias("month"),
    )
)

## 3. Filter Data

1. `type == 0`: hanya kebakaran vegetasi; tipe 1 (gunung api),
   2 (sumber statis), dan 3 (lepas pantai) bukan kebakaran hutan/lahan.
2. `confidence != "l"`: deteksi berkeyakinan rendah lebih rawan
   false alarm.
3. `date_local <= 2024-12-31`: konversi ke waktu lokal menggeser
   deteksi malam 31 Desember 2024 ke 1 Januari 2025; titik ini dibuang
   agar cakupan data tetap 2012–2024.

Kolom konstan (`satellite`, `instrument`, `version`) dan `type`
(konstan setelah filter) dihapus.

In [4]:
lf_type = lf_time.filter(pl.col("type") == 0)
lf_conf = lf_type.filter(pl.col("confidence") != "l")
lf_date = lf_conf.filter(pl.col("date_local") <= pl.date(2024, 12, 31))

log = pl.DataFrame(
    [
        ("raw", n_raw),
        ("type == 0 (vegetasi)", lf_type.select(pl.len()).collect().item()),
        ("confidence != low", lf_conf.select(pl.len()).collect().item()),
        ("date_local <= 2024-12-31", lf_date.select(pl.len()).collect().item()),
    ],
    schema=["tahap", "n_rows"],
    orient="row",
).with_columns(
    (pl.col("n_rows").shift(1) - pl.col("n_rows")).alias("dibuang")
)

lf_drop = lf_date.drop(["satellite", "instrument", "version", "type"])
log

tahap,n_rows,dibuang
str,i64,i64
"""raw""",3418464,null
"""type == 0 (vegetasi)""",3267691,150773
"""confidence != low""",3060782,206909
"""date_local <= 2024-12-31""",3060774,8


## 4. Penandaan Outlier dan Tahun Tidak Lengkap

Distribusi `frp` sangat menceng ke kanan. Nilai ekstrem kemungkinan
kebakaran besar yang nyata, sehingga **ditandai (flag), bukan dihapus**.
Metode IQR diterapkan pada skala asli dan dibandingkan dengan skala log.

Tahun 2012 ditandai karena data baru dimulai 20 Januari 2012.

In [5]:
q1, q3 = lf_drop.select(
    pl.col("frp").quantile(0.25).alias("q1"),
    pl.col("frp").quantile(0.75).alias("q3"),
).collect().row(0)
upper = q3 + 1.5 * (q3 - q1)

lq1, lq3 = lf_drop.select(
    pl.col("frp").log1p().quantile(0.25).alias("lq1"),
    pl.col("frp").log1p().quantile(0.75).alias("lq3"),
).collect().row(0)
upper_log = lq3 + 1.5 * (lq3 - lq1)

lf_clean = lf_drop.with_columns(
    (pl.col("frp") > upper).alias("frp_outlier"),
    (pl.col("year") == 2012).alias("partial_year"),
)

pct = lf_clean.select(
    pl.col("frp_outlier").mean().alias("pct_iqr_asli"),
    (pl.col("frp").log1p() > upper_log).mean().alias("pct_iqr_log"),
).collect()

print(f"Batas atas IQR (skala asli): {upper:.2f} MW")
print(f"Batas atas IQR (skala log) : {pl.Series([upper_log]).exp().item() - 1:.2f} MW")
pct

Batas atas IQR (skala asli): 17.85 MW
Batas atas IQR (skala log) : 44.97 MW


pct_iqr_asli,pct_iqr_log
f64,f64
0.088537,0.018849


## 5. Validasi dan Penyimpanan

In [6]:
df = lf_clean.collect()

assert df["latitude"].is_between(-11.5, 6.5).all(), "Lintang di luar Indonesia"
assert df["longitude"].is_between(94.5, 141.5).all(), "Bujur di luar Indonesia"
assert (df["frp"] >= 0).all(), "FRP negatif"
assert df["dt_utc"].null_count() == 0, "Datetime gagal dibentuk"
assert df["year"].is_between(2012, 2024).all(), "Tahun di luar cakupan"
assert set(df["confidence"].unique()) <= {"n", "h"}, "Confidence low masih ada"
assert df.is_duplicated().sum() == 0, "Ada baris duplikat"
print("Semua validasi lolos.")

df.write_parquet(OUT_PATH)
csv_mb = sum(f.stat().st_size for f in (ROOT / "data/raw/firms").glob("*.csv")) / 1e6
parquet_mb = OUT_PATH.stat().st_size / 1e6
print(f"Shape: {df.shape}")
print(f"CSV: {csv_mb:,.1f} MB → Parquet: {parquet_mb:,.1f} MB ({csv_mb / parquet_mb:.1f}x lebih kecil)")

Semua validasi lolos.
Shape: (3060774, 18)
CSV: 270.3 MB → Parquet: 60.3 MB (4.5x lebih kecil)


## 6. Profiling Ulang dengan DuckDB `SUMMARIZE`

In [7]:
duckdb.sql(f"SUMMARIZE SELECT * FROM '{OUT_PATH}'").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,latitude,DOUBLE,-10.97676,5.8782,1019804,-2.625583686672061,3.3420290277242914,-3.669927215403826,-2.43654195218387,-0.17871677015394577,3060774,0.0
1,longitude,DOUBLE,95.01992,140.9774,1664561,112.70830595248442,10.188907199364722,104.31854848858544,111.87893296545265,117.1392931773142,3060774,0.0
2,bright_ti4,DOUBLE,295.0,367.0,5957,331.50171033536145,16.69775986253654,318.7971095295911,334.4399602229474,342.0949894975036,3060774,0.0
3,scan,DOUBLE,0.32,0.8,47,0.45581250036750354,0.08276424303500943,0.39,0.43283759692932805,0.5013224424644225,3060774,0.0
4,track,DOUBLE,0.36,0.78,43,0.470509289480333,0.10870435658312522,0.37822709984903485,0.4393268102385832,0.5465299415295997,3060774,0.0
5,acq_date,DATE,2012-01-20,2024-12-31,5049,2016-12-12 09:42:03.279406,NaN,2014-09-01,2015-10-08,2019-08-30,3060774,0.0
6,acq_time,VARCHAR,0308,2005,525,NaN,NaN,NaN,NaN,NaN,3060774,0.0
7,confidence,VARCHAR,h,n,2,NaN,NaN,NaN,NaN,NaN,3060774,0.0
8,bright_ti5,DOUBLE,150.0,378.18,9107,293.81528872435644,8.353251069437842,289.8377336962345,293.95199891930827,299.14529930983065,3060774,0.0
9,frp,DOUBLE,0.0,1951.3,13646,8.089155118933977,14.153989662746355,2.3542733453359186,4.648856120406738,8.554748094075244,3060774,0.0


## 7. Verifikasi Pergeseran Tanggal Lokal

In [8]:
(df.group_by("daynight")
   .agg((pl.col("date_local") != pl.col("acq_date")).mean().alias("pct_tanggal_bergeser"))
   .sort("daynight"))

daynight,pct_tanggal_bergeser
str,f64
"""D""",0.0
"""N""",1.0


## 8. Ringkasan Cleaning

### Alur dan jumlah baris
| Tahap | Baris | Dibuang |
|---|---:|---:|
| Raw | 3.418.464 | – |
| `type == 0` | 3.267.691 | 150.773 |
| `confidence != "l"` | 3.060.782 | 206.909 |
| `date_local <= 2024-12-31` | 3.060.774 | 8 |

Data akhir berisi **3.060.774 baris (89,5% dari data mentah)**.

### Transformasi
- Schema ditetapkan eksplisit untuk menghindari perbedaan inferensi
  tipe antara Polars dan DuckDB.
- Kolom waktu lokal terbukti diperlukan: **100% deteksi malam** bergeser
  ke tanggal berikutnya, sedangkan deteksi siang tidak bergeser.
- Konversi waktu lokal memunculkan 8 titik bertanggal 1 Januari 2025
  yang kemudian dibuang.
- Kolom dihapus: `satellite`, `instrument`, `version`, `type` (konstan).
- Kolom ditambahkan: `dt_utc`, `dt_local_solar`, `date_local`, `year`,
  `month`, `frp_outlier`, `partial_year`.

### Outlier
- Batas atas IQR `frp` pada skala asli: **17,85 MW**, menandai
  **8,85%** titik.
- Pada skala log, batas atas menjadi **44,97 MW** dan proporsi yang
  ditandai turun menjadi **1,88%**. IQR pada skala asli kurang tepat
  untuk distribusi yang sangat menceng karena menandai terlalu banyak
  titik yang sebenarnya wajar.
- Outlier ditandai, tidak dihapus, karena mewakili kebakaran paling
  intens yang justru penting dalam analisis.

### Temuan tambahan
- Setelah filter confidence, `bright_ti4` minimum naik dari 208 K
  menjadi 295 K: deteksi bersinyal panas lemah seluruhnya berkategori
  confidence rendah.
- Nilai `approx_unique` pada DuckDB `SUMMARIZE` bersifat aproksimasi
  (HyperLogLog), sehingga jumlah nilai unik eksak dihitung dengan Polars.

### Efisiensi penyimpanan
- CSV **270,3 MB** → Parquet **60,3 MB** (**4,5x lebih kecil**),
  meskipun jumlah kolom bertambah.

### Validasi
Seluruh pemeriksaan lolos: koordinat dalam wilayah Indonesia, FRP
non-negatif, datetime lengkap, tahun 2012–2024, tidak ada confidence
rendah, dan tidak ada duplikat.